# Visão da Analytical Base Table (ABT)

Este notebook valida `Dados/abt.csv`: granularidade, blocos de features, cobertura dos históricos, qualidade e relação das variáveis agregadas com o TARGET.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if ROOT.name in {"DataPipeline", "Model"}:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from MLOps import storage

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

In [ ]:
abt=storage.read_csv("Dados/abt.csv", low_memory=False)
print(f"ABT: {abt.shape[0]:,} linhas x {abt.shape[1]:,} colunas")
print(f"IDs únicos: {abt['SK_ID_CURR'].nunique():,}")
print(f"Duplicidades de ID: {abt['SK_ID_CURR'].duplicated().sum():,}")
print(f"Taxa de default: {abt['TARGET'].mean():.2%}")

## 1. Composição por bloco

In [ ]:
blocks={
 'Identificador/Alvo':[c for c in abt if c in ['SK_ID_CURR','TARGET']],
 'Bureau':[c for c in abt if c.startswith('BUREAU_')],
 'Previous':[c for c in abt if c.startswith('PREV_')],
 'Installments':[c for c in abt if c.startswith('INSTAL_')],
 'Razões':[c for c in abt if c in ['CREDIT_INCOME_RATIO','ANNUITY_INCOME_RATIO','ANNUITY_CREDIT_RATIO','AGE_YEARS','EMPLOYED_YEARS']],
}
assigned=set(sum(blocks.values(),[]))
blocks['Application']=[c for c in abt if c not in assigned]
pd.DataFrame({'bloco':blocks.keys(),'quantidade':[len(v) for v in blocks.values()]}).set_index('bloco').plot(kind='bar',legend=False,title='Colunas por bloco'); plt.ylabel('nº colunas'); plt.show()
display(pd.DataFrame({'bloco':blocks.keys(),'quantidade':[len(v) for v in blocks.values()]}))

## 2. Cobertura dos históricos

In [ ]:
coverage={}
for prefix,anchor in [('BUREAU_','BUREAU_CREDIT_COUNT'),('PREV_','PREV_APP_COUNT'),('INSTAL_','INSTAL_COUNT')]:
    if anchor in abt:
        coverage[prefix.rstrip('_')]=float((abt[anchor]>0).mean())
display(pd.Series(coverage,name='coverage').to_frame().style.format('{:.2%}'))

## 3. Qualidade da ABT

In [ ]:
quality=pd.DataFrame({
 'dtype':abt.dtypes.astype(str),
 'missing_pct':abt.isna().mean()*100,
 'n_unique':abt.nunique(dropna=False)
}).sort_values('missing_pct',ascending=False)
display(quality.head(30))

## 4. Features comportamentais de parcelas

In [ ]:
inst=[c for c in ['INSTAL_DELAY_MEAN','INSTAL_DPD_MAX','INSTAL_LATE_PAYMENT_RATIO','INSTAL_LATE_30D_RATIO','INSTAL_PARTIAL_PAYMENT_RATIO','INSTAL_PAYMENT_RATIO_MEAN'] if c in abt]
if inst:
    display(abt.groupby('TARGET')[inst].mean().T)
else:
    print('Nenhuma feature INSTAL_* encontrada.')

## 5. Relação numérica com o TARGET

In [ ]:
num=abt.select_dtypes(include=np.number)
corr=num.corr(numeric_only=True)['TARGET'].drop('TARGET').sort_values(key=lambda s:s.abs(),ascending=False)
display(corr.head(30).to_frame('corr_target'))
corr.head(20).sort_values().plot(kind='barh',title='Top correlações lineares com TARGET'); plt.show()

## 6. Distribuições das razões financeiras

In [ ]:
ratios=[c for c in ['CREDIT_INCOME_RATIO','ANNUITY_INCOME_RATIO','ANNUITY_CREDIT_RATIO'] if c in abt]
for c in ratios:
    q=abt[c].quantile(.99)
    abt[abt[c]<=q].boxplot(column=c,by='TARGET'); plt.suptitle(''); plt.title(f'{c} por TARGET (até p99)'); plt.show()

## 7. Checklist de prontidão

- Uma linha por `SK_ID_CURR`.
- `TARGET` preservado e sem duplicidade de chave.
- Features históricas ausentes preenchidas com zero, representando ausência de histórico.
- Nulos restantes tratados dentro do pipeline Scikit-Learn, evitando leakage.
- Blocos `BUREAU_*`, `PREV_*` e `INSTAL_*` disponíveis para avaliação de ganho incremental.
- ABT pronta para o split treino/holdout e seleção por validação cruzada.